In [2]:
import os

import matplotlib.pyplot as plt
import mne
import numpy as np
from nilearn.plotting import plot_design_matrix

import mne_nirs
from mne_nirs.channels import get_long_channels, get_short_channels, picks_pair_to_idx
from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.statistics import run_glm
from mne.preprocessing.nirs import (
    optical_density,
    temporal_derivative_distribution_repair,
)


#https://mne.tools/stable/auto_tutorials/preprocessing/70_fnirs_processing.html#sphx-glr-auto-tutorials-preprocessing-70-fnirs-processing-py im you want to look

In [ ]:
# Path to a specific participant folder
# Specify the path to your NIRx data directory
fnirs_cw_amplitude_dir = ""

# Load NIRx data
raw = mne.io.read_raw_nirx(fnirs_cw_amplitude_dir, verbose=True)
raw.load_data()

Loading C:/Users/sasha/Documents/Study Stuff/fnirsdata
Reading 0 ... 26197  =      0.000 ...  2575.270 secs...


C:\Users\sasha\AppData\Local\Temp\ipykernel_14996\2180230261.py:4: RuntimeWarning: MNE has not been tested with Aurora version 2023.9.3-1-g01ca56fc
  raw = mne.io.read_raw_nirx(fnirs_cw_amplitude_dir, verbose=True)


<RawNIRX | fnirsdata, 30 x 26198 (2575.4 s), ~6.0 MiB, data loaded>

In [4]:
raw.annotations.rename(
    {
        "2.0": "Scene",
        "3.0": "Scene",
        "4.0": "End_Game",
        "6.0": "Rest",
        "7.0": "Rest",
        "8.0": "Measurement",
        "9.0": "Measurement",
    }
)
raw.annotations.delete(raw.annotations.description == "End_Game")

#With actual data this needs to be changed (since it should have less tags)

In [ ]:
#setting up sources

front_source = [1,2,3,4]
front_dectector = [1,2,3,4]

back_source = [5,6,7,8]
back_dectector = [5,6,7,8]

#Splitting channels into frontal and posterior regions based on source/detector numbers 

def get_region_channels(raw, sources, detectors):
    """Return channel names where both source AND detector are in the given sets."""
    matching = []
    for ch in raw.ch_names:
        # Channels names are S1_D1 etc
        parts = ch.split(" ")[0]  # strip hbo/hbr suffix
        if not (parts.startswith("S") and "_D" in parts):
            continue
        s_num = int(parts.split("_D")[0][1:])  # extract source number
        d_num = int(parts.split("_D")[1])       # extract detector number
        if s_num in sources and d_num in detectors:
            matching.append(ch)
    return matching

frontal_chs   = get_region_channels(raw, front_source, front_dectector)
posterior_chs = get_region_channels(raw, back_source, back_dectector)

raw_frontal   = raw.copy().pick(frontal_chs)
raw_posterior = raw.copy().pick(posterior_chs)

#Making sure things make sense

print(f"Frontal channels:   {len(frontal_chs)}")
print(f"Posterior channels: {len(posterior_chs)}")


print("Frontal channels:", frontal_chs)
print("Posterior channels:", posterior_chs)

Frontal channels:   14
Posterior channels: 16
Frontal channels: ['S1_D1 760', 'S1_D1 850', 'S1_D2 760', 'S1_D2 850', 'S1_D3 760', 'S1_D3 850', 'S2_D2 760', 'S2_D2 850', 'S3_D2 760', 'S3_D2 850', 'S4_D1 760', 'S4_D1 850', 'S4_D4 760', 'S4_D4 850']
Posterior channels: ['S5_D5 760', 'S5_D5 850', 'S6_D6 760', 'S6_D6 850', 'S6_D7 760', 'S6_D7 850', 'S6_D8 760', 'S6_D8 850', 'S7_D6 760', 'S7_D6 850', 'S7_D7 760', 'S7_D7 850', 'S8_D6 760', 'S8_D6 850', 'S8_D8 760', 'S8_D8 850']


In [8]:
def preprocess_region(raw_region, region_name):
    
    print(f"\n--- Processing {region_name} ---")

    # Convert to optical density
    raw_od = optical_density(raw_region)

    #Temporal Derivative Distribution Repair
    raw_od_tddr = temporal_derivative_distribution_repair(raw_od)

    # SCI
    sci = mne.preprocessing.nirs.scalp_coupling_index(raw_od_tddr)
    bad_chs = [
        raw_od_tddr.ch_names[i]
        for i, s in enumerate(sci)
        if s < 0.5
    ]
    if bad_chs:
        print(f"Bad channels (SCI < 0.5): {bad_chs}")
        raw_od_tddr.info["bads"] = bad_chs
    else:
        print(f"All channels passed SCI check.")

    raw_haemo = mne.preprocessing.nirs.beer_lambert_law(raw_od_tddr, ppf=0.1)

    print(f"Channels after processing: {raw_haemo.ch_names}")
    return raw_haemo

haemo_frontal   = preprocess_region(raw_frontal, "Frontal")
haemo_posterior = preprocess_region(raw_posterior, "Posterior")

def split_hbo_hbr(raw_haemo, region_name):
    hbo = raw_haemo.copy().pick(picks="hbo")
    hbr = raw_haemo.copy().pick(picks="hbr")
    print(f"{region_name} — HbO channels: {len(hbo.ch_names)}, "
          f"HbR channels: {len(hbr.ch_names)}")
    return hbo, hbr

frontal_hbo,   frontal_hbr   = split_hbo_hbr(haemo_frontal,   "Frontal")
posterior_hbo, posterior_hbr = split_hbo_hbr(haemo_posterior, "Posterior")


--- Processing Frontal ---
Bad channels (SCI < 0.5): ['S1_D1 760', 'S1_D1 850', 'S1_D3 760', 'S1_D3 850']
Channels after processing: ['S1_D1 hbo', 'S1_D1 hbr', 'S1_D2 hbo', 'S1_D2 hbr', 'S1_D3 hbo', 'S1_D3 hbr', 'S2_D2 hbo', 'S2_D2 hbr', 'S3_D2 hbo', 'S3_D2 hbr', 'S4_D1 hbo', 'S4_D1 hbr', 'S4_D4 hbo', 'S4_D4 hbr']

--- Processing Posterior ---
All channels passed SCI check.
Channels after processing: ['S5_D5 hbo', 'S5_D5 hbr', 'S6_D6 hbo', 'S6_D6 hbr', 'S6_D7 hbo', 'S6_D7 hbr', 'S6_D8 hbo', 'S6_D8 hbr', 'S7_D6 hbo', 'S7_D6 hbr', 'S7_D7 hbo', 'S7_D7 hbr', 'S8_D6 hbo', 'S8_D6 hbr', 'S8_D8 hbo', 'S8_D8 hbr']
Frontal — HbO channels: 7, HbR channels: 7
Posterior — HbO channels: 8, HbR channels: 8
